# 03 — Results and robustness charts

**Goal:** compare the trained models and see how their scores drop when the photos are damaged
(blur, JPEG, downscale, noise, screen — see notebook 02).

**This notebook does not run any model.** It only reads score files that were already saved by
`python -m deepfake_kyc.evaluate` (or the last cell of notebook 05). Each of those runs saves one file:

```
outputs/<experiment>/eval_test.csv      e.g. outputs/02_effnet_frozen/eval_test.csv
```

Each row of that file is one test condition: `clean` (undamaged photos) or one kind of damage at severity 1–5.

**How to use:** run the cells from top to bottom (Shift+Enter). Works on your own computer, in Google Colab and on Kaggle. No GPU needed.

### Running on Kaggle? One-time setup

In the notebook editor, open the right-hand panel:

- **Settings → Internet → On.** Needed to download the project code from GitHub. Kaggle asks you to verify your phone once.
- **Add Input → Datasets →** search **"140k Real and Fake Faces"** (by xhlulu) → **Add**. These are the images;
  Kaggle attaches them read-only under `/kaggle/input/`, so nothing is downloaded or unzipped.
- **Add Input → Your Work →** your saved notebook 05 (after **Save Version** there). This brings in its trained
  models (`outputs/<experiment>/best.pt`) and score files, read-only.
- Only for Step 1b (scoring models): **Settings → Accelerator → GPU T4 x2** (or P100). Makes the model steps much faster (free, about 30 hours a week).

The code finds everything by itself: `ON_KAGGLE` in Step 1 is `True` when running on Kaggle.
Files you create go to `/kaggle/working/`; they are kept only when you click **Save Version**.

## Step 1 — Find the project and the results folders

**What this does, in plain words:**

1. Loads the Python tools we need (`pandas` for tables, `matplotlib` for charts).
2. Finds the project folder. On Colab and Kaggle it downloads (clones) it from GitHub first.
3. Makes a list of folders to search for results:
   - `<project>/outputs/` — where `train.py` and `evaluate.py` save by default.
   - On Colab only: your Google Drive folder `MyDrive/deepfake_kyc_outputs/`, where notebook 05 saves
     on Colab. Colab will ask you to allow access to your Drive.
   - On Kaggle only: the `outputs/` folders of notebooks you attached with **Add Input → Your Work**
     (for example your saved notebook 05). They are read-only.
4. Finds every `eval_*.csv` file inside those folders. If the same file is found twice, the copy in this
   notebook's own `outputs/` folder wins, because it is the newest.

In [ ]:
# 1. Tools we need
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from IPython.display import display

# Where are we running? Each platform keeps files in a different place.
ON_COLAB = "google.colab" in sys.modules           # Google Colab
ON_KAGGLE = Path("/kaggle/input").exists()         # Kaggle notebook

# 2. Find the project folder: look in the current folder and every folder above it.
PROJECT_ROOT = None
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / "src" / "deepfake_kyc").is_dir():
        PROJECT_ROOT = folder
        break

# Not found (for example on Colab or Kaggle)? Download the project from GitHub into the current folder.
# (On Kaggle this needs Internet switched on: Settings -> Internet -> On.)
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd() / "UOT_SCS_3546_041_Deep-Learning"
    if not PROJECT_ROOT.exists():
        subprocess.run(["git", "clone", "https://github.com/msondhi/UOT_SCS_3546_041_Deep-Learning.git",
                        str(PROJECT_ROOT)], check=True)
print("Project folder:", PROJECT_ROOT)

# 3. Read the settings file to know where the test images are (only needed for Step 1b).
sys.path.insert(0, str(PROJECT_ROOT / "src"))   # lets Python import the project's own code
from deepfake_kyc.config import load_config

cfg = load_config(PROJECT_ROOT / "configs" / "base.yaml")
DATA_ROOT = PROJECT_ROOT / cfg["data"]["root"]     # folder with train/ valid/ test/

# On Kaggle the dataset is attached read-only under /kaggle/input (Add Input -> "140k Real and Fake Faces"),
# so we use it from there instead of data/raw/. Nothing needs to be downloaded or unzipped.
def find_kaggle_dataset():
    """Look inside /kaggle/input for the folder called real-vs-fake that has train/ valid/ test/."""
    for depth in range(1, 5):                                  # look 1, 2, 3, then 4 folders deep
        pattern = "/".join(["*"] * depth) + "/real-vs-fake"    # e.g. "*/*/real-vs-fake"
        for folder in Path("/kaggle/input").glob(pattern):
            if (folder / "train").is_dir():
                return folder
    return None

if ON_KAGGLE and find_kaggle_dataset() is not None:
    DATA_ROOT = find_kaggle_dataset()
print("Images folder:", DATA_ROOT)

# 4. Folders to search for results.
RESULTS_FOLDERS = []
if ON_KAGGLE:
    # Outputs of your other Kaggle notebooks that you attached with Add Input (read-only),
    # e.g. notebook 05 after "Save Version". They look like /kaggle/input/<notebook>/.../outputs/
    for depth in range(1, 5):
        RESULTS_FOLDERS += sorted(Path("/kaggle/input").glob("/".join(["*"] * depth) + "/outputs"))

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")      # asks you to allow access to your Drive
    RESULTS_FOLDERS.append(Path("/content/drive/MyDrive/deepfake_kyc_outputs"))

RESULTS_FOLDERS.append(PROJECT_ROOT / "outputs")   # last, so a new score made here wins

# 5. Find every eval_*.csv file: <results folder>/<experiment>/eval_*.csv
def find_eval_files():
    found = {}
    for results_folder in RESULTS_FOLDERS:
        for file in sorted(results_folder.glob("*/eval_*.csv")):
            found[(file.parent.name, file.name)] = file   # a later folder replaces an older copy
    return sorted(found.values())

eval_files = find_eval_files()

print(f"Found {len(eval_files)} score file(s):")
for file in eval_files:
    print("  ", file)

## Step 1b (optional) — Run `evaluate.py` from here

Skip this if Step 1 already found the score files you want.

**What `evaluate.py` does, in plain words:**

1. Opens a trained model (`best.pt`) and the settings saved inside it.
2. Makes a list of test conditions:
   - always `clean`: the normal test photos, undamaged (saved as severity 0);
   - only if you ask for damage: each of the 5 kinds of damage at severities 1–5, which adds 25 more conditions.
3. For each condition, it runs the model on **all** test photos (damaged on the fly, the files on disk are not changed)
   and calculates accuracy, ROC AUC, FAR, FRR, …
4. Saves one row per condition into `eval_test.csv`, next to `best.pt`.

So a file with **1 row** was tested on clean photos only. A file with **26 rows** was tested on clean + all damage.

**How this cell decides what to test (`INCLUDE_DAMAGE = "auto"`):** every model is trained on real **and** fake faces.
The difference is whether damage was added during training. The cell reads the `config.yaml` saved next to each `best.pt`:

| How the model was trained | `augment.degradation` | What this cell tests |
|---|---|---|
| Clean + randomly damaged photos (`01`–`05`, the project default) | `true` | clean + all damage (26 tests) |
| Clean photos only (`06_effnet_clean`, the control) | `false` | clean test photos only (1 test) |

You can override it with `INCLUDE_DAMAGE = True` or `False`.

**Other controls:** which models (`CHECKPOINTS`) and how many test photos (`TEST_PHOTOS`, fewer = faster).
It needs PyTorch and a GPU is much faster (on Colab: Runtime → Change runtime type → T4 GPU).
It overwrites each model's existing `eval_test.csv`. Change `RUN_EVALUATION` to `True` to run it.

In [ ]:
RUN_EVALUATION = False      # change to True to run evaluate.py
INCLUDE_DAMAGE = "auto"     # "auto" = test the way the model was trained; True = always add damage; False = clean only
TEST_PHOTOS = 2000          # use this many test photos to save time; None = all 20,000
CHECKPOINTS = None          # None = every best.pt found; or a list, e.g. [PROJECT_ROOT / "outputs" / "02_effnet_frozen" / "best.pt"]

if RUN_EVALUATION:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))   # lets Python import the project's own code
    from deepfake_kyc.evaluate import main as run_evaluate

    if CHECKPOINTS is None:
        CHECKPOINTS = []
        for results_folder in RESULTS_FOLDERS:
            CHECKPOINTS += sorted(results_folder.glob("*/best.pt"))
    if not CHECKPOINTS:
        print("No best.pt found: train a model first (notebook 05 or train.py).")

    for checkpoint in CHECKPOINTS:
        # How was this model trained? The training settings are saved next to best.pt.
        settings = yaml.safe_load((checkpoint.parent / "config.yaml").read_text())
        trained_with_damage = settings.get("augment", {}).get("degradation", False)
        test_damage = trained_with_damage if INCLUDE_DAMAGE == "auto" else INCLUDE_DAMAGE

        print("=" * 70)
        print("Model:     ", checkpoint.parent.name)
        print("Trained on:", "clean + randomly damaged photos" if trained_with_damage else "clean photos only")
        print("Testing on:", "clean + damaged photos (26 tests)" if test_damage else "clean photos only (1 test)")

        # These are the same words you would type after "python -m deepfake_kyc.evaluate" in a terminal.
        # data.root tells evaluate.py where the test images are on THIS computer (local, Colab or Kaggle).
        arguments = ["--checkpoint", str(checkpoint), "--set", "data.num_workers=0", f"data.root={DATA_ROOT.as_posix()}"]
        if TEST_PHOTOS is not None:
            arguments.append(f"data.test_subset={TEST_PHOTOS}")
        if test_damage:
            arguments += ["--degradation", "all"]

        # On Kaggle, models from another notebook are read-only, so save the score in this notebook's outputs/.
        if not os.access(checkpoint.parent, os.W_OK):
            score_folder = PROJECT_ROOT / "outputs" / checkpoint.parent.name
            score_folder.mkdir(parents=True, exist_ok=True)
            (score_folder / "config.yaml").write_text((checkpoint.parent / "config.yaml").read_text())
            arguments += ["--output", str(score_folder / "eval_test.csv")]
        print("Running:    python -m deepfake_kyc.evaluate", " ".join(arguments))
        run_evaluate(arguments)

    # Look for score files again so the new one is included.
    eval_files = find_eval_files()
    print(f"Now {len(eval_files)} score file(s) found.")
else:
    print("Skipped. Change RUN_EVALUATION to True to run evaluate.py from this notebook.")

**Prefer a terminal?** The same thing from the project folder:

```bash
python -m deepfake_kyc.evaluate --checkpoint outputs/02_effnet_frozen/best.pt                     # clean only
python -m deepfake_kyc.evaluate --checkpoint outputs/02_effnet_frozen/best.pt --degradation all   # clean + damage
```

## Step 2 — Put all score files into one table

**What this does:** reads every file and stacks them into one table called `results`.
It adds a `run` column so we know which experiment each row came from, e.g. `02_effnet_frozen (test)`.
The folder name is the experiment name; the part after `eval_` in the file name is the test set (`test` or e.g. `ood`).

In [ ]:
tables = []
for file in eval_files:
    table = pd.read_csv(file)
    experiment = file.parent.name                    # e.g. 02_effnet_frozen
    test_set = file.stem.replace("eval_", "")        # e.g. test
    table["run"] = f"{experiment} ({test_set})"

    # How was this model trained? Read the settings saved next to the score file.
    settings_file = file.parent / "config.yaml"
    settings = yaml.safe_load(settings_file.read_text()) if settings_file.exists() else {}
    trained_with_damage = settings.get("augment", {}).get("degradation", False)
    table["trained_on"] = "clean + damaged photos" if trained_with_damage else "clean photos only"
    tables.append(table)

if tables:
    results = pd.concat(tables, ignore_index=True)
    print(f"{len(results)} rows from {len(tables)} file(s). First rows:")
    display(results.head())
else:
    results = pd.DataFrame()
    print("No score files yet: run Step 1b, or evaluate.py in a terminal.")

## Step 2b — What did `evaluate.py` test in each file?

**What this does:** for each run, says how the model was **trained** (clean only, or clean + damaged photos)
and what it was **tested** on (clean only, or clean + which kinds of damage and severities).
Use this to understand why a run does or does not appear in the charts of Steps 4–5.

In [ ]:
if not results.empty:
    for run in results["run"].unique():
        rows = results[results["run"] == run]
        damaged = rows[rows["condition"] != "clean"]
        print(f"{run}: {len(rows)} test condition(s)")
        print(f"   trained on:     {rows['trained_on'].iloc[0]}")
        if (rows["condition"] == "clean").any():
            print("   clean photos:   yes")
        if damaged.empty:
            print("   damaged photos: no  -> clean only, so this run is skipped in Steps 4-5")
        else:
            for damage in sorted(damaged["condition"].unique()):
                severities = sorted(damaged.loc[damaged["condition"] == damage, "severity"])
                print(f"   damaged photos: {damage:<9} severities {severities}")
        print()
else:
    print("Skipped: no results loaded.")

## Step 3 — Clean test results: which model is best?

**What this does:** shows one row per run, using only the `clean` (undamaged) photos, best first.

**What the columns mean** (fake = the thing we want to catch):

| Column | Plain meaning | Better when |
|---|---|---|
| `accuracy` | share of all photos labelled correctly | higher |
| `precision` | of the photos flagged as fake, how many really were fake | higher |
| `recall` | of all the fakes, how many were caught | higher |
| `f1` | one number that balances precision and recall | higher |
| `roc_auc` | how well the model ranks fakes above reals, at any threshold (1.0 = perfect, 0.5 = guessing) | higher |
| `far` | **False Acceptance Rate**: share of fakes that got through as real (the fraud risk) | lower |
| `frr` | **False Rejection Rate**: share of real customers wrongly flagged as fake (the annoyed-customer risk) | lower |

In [ ]:
if not results.empty:
    clean = results[results["condition"] == "clean"]
    clean = clean.sort_values("roc_auc", ascending=False)
    columns = ["run", "accuracy", "precision", "recall", "f1", "roc_auc", "far", "frr"]
    display(clean[columns].round(3))
else:
    print("Skipped: no results loaded.")

## Step 4 — Robustness charts: how much do scores drop on damaged photos?

**What this does:** draws one small chart per kind of damage. The x-axis is the severity
(0 = clean photos, 1 = a little damage … 5 = a lot). Each line is one run.

**How to read it:** a line that stays flat means the model copes well with that damage.
A line that falls steeply means the model breaks on poor-quality uploads.

**Try it:** change `metric` to `"accuracy"`, `"far"` or `"frr"` and run the cell again.
For `far` and `frr`, lower is better, so a rising line is the bad sign.

In [ ]:
metric = "roc_auc"    # try "accuracy", "far" or "frr"

if not results.empty and (results["condition"] != "clean").any():
    damage_names = sorted(results.loc[results["condition"] != "clean", "condition"].unique())

    fig, axes = plt.subplots(1, len(damage_names), figsize=(4 * len(damage_names), 3.5),
                             sharey=True, squeeze=False)
    for ax, damage in zip(axes[0], damage_names):
        for run in results["run"].unique():
            # this run's rows for this damage, plus its clean row as severity 0
            rows = results[(results["run"] == run) & results["condition"].isin(["clean", damage])]
            rows = rows.sort_values("severity")
            if len(rows) > 1:
                ax.plot(rows["severity"], rows[metric], marker="o", label=run)
        ax.set_title(damage)
        ax.set_xlabel("severity (0 = clean)")
        ax.set_xticks(range(6))
    axes[0, 0].set_ylabel(metric)
    axes[0, -1].legend(fontsize=8)
    fig.suptitle(f"{metric} as the damage gets worse")
    plt.tight_layout()
    plt.show()
else:
    print("Skipped: no damaged-photo results yet. Run evaluate with --degradation all.")

## Step 5 — Summary: clean score vs. worst-case score

**What this does:** for each run and each kind of damage, compares the clean score with the score at severity 5,
and shows how much it dropped. This gives numbers you can quote in the report,
e.g. *"blur at severity 5 lowers ROC AUC by 0.12"*.

In [ ]:
if not results.empty and (results["condition"] != "clean").any():
    summary_rows = []
    for run in results["run"].unique():
        run_rows = results[results["run"] == run]
        clean_rows = run_rows[run_rows["condition"] == "clean"]
        if clean_rows.empty:
            continue
        clean_score = clean_rows[metric].iloc[0]
        for damage in sorted(run_rows.loc[run_rows["condition"] != "clean", "condition"].unique()):
            damage_rows = run_rows[run_rows["condition"] == damage]
            worst = damage_rows.loc[damage_rows["severity"].idxmax()]   # highest severity tested
            summary_rows.append({
                "run": run,
                "damage": damage,
                f"clean {metric}": clean_score,
                f"{metric} at severity {int(worst['severity'])}": worst[metric],
                "change": worst[metric] - clean_score,
            })
    display(pd.DataFrame(summary_rows).round(3))
else:
    print("Skipped: no damaged-photo results yet.")

## Step 6 — One overall score per model (clean + damaged mix)

**What this does:** gives each model one score for each column, matching how it was trained:

- Model trained on **clean + damaged** photos and tested on both: the **average** over all its
  26 tests (1 clean + 5 kinds of damage × 5 severities). This is like scoring it on a test set where
  photos come in every quality, from perfect to badly damaged.
- Model trained on **clean photos only** and tested clean only: its clean score.

The `tested on` column says which one you are looking at. Only compare models that have the same `tested on` value:
an average that includes damaged photos is naturally lower than a clean-only score.

In [ ]:
if not results.empty:
    overall_rows = []
    for run in results["run"].unique():
        run_rows = results[results["run"] == run]
        has_damage = (run_rows["condition"] != "clean").any()
        overall_rows.append({
            "run": run,
            "trained on": run_rows["trained_on"].iloc[0],
            "tested on": f"clean + damaged ({len(run_rows)} tests)" if has_damage else "clean only (1 test)",
            "accuracy": run_rows["accuracy"].mean(),   # mean = average over all tests of this run
            "roc_auc": run_rows["roc_auc"].mean(),
            "far": run_rows["far"].mean(),
            "frr": run_rows["frr"].mean(),
        })
    overall = pd.DataFrame(overall_rows).sort_values("roc_auc", ascending=False)
    display(overall.round(3))
else:
    print("Skipped: no results loaded.")

## Next

[04_error_analysis.ipynb](04_error_analysis.ipynb) opens one trained model, finds the photos it got wrong,
and shows heatmaps (Grad-CAM) of where the model was looking.

## Quick notes to remember

- These numbers come from the **test** set. Use them only for the final comparison, not to tune settings.
- `far` matters most for fraud (fakes getting through); `frr` matters most for customer experience.
- A run without `--degradation all` only has a clean row, so it appears in Steps 3 and 6 but not in Steps 4–5.
- Which models train on damaged photos is set in the config: `augment.degradation: true` in `configs/base.yaml`,
  so `01`–`05` train on clean + damaged photos. `06_effnet_clean` is the control trained on clean photos only.
- To show that damage training helps, also test E6 on damaged photos (`INCLUDE_DAMAGE = True` in Step 1b)
  and compare it with E3 in the Step 4 charts.
- On Colab, results disappear with the session unless they are in Google Drive.